In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [13]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes, usually you can join even if you just discovered the course — but it depends on the course’s enrollment rules and whether registration is still open.

If you want, I can help you phrase a quick message like this:

> Hi, I just discovered the course and I’m very interested in joining. Is it still possible to enroll now?

If you’d like, I can also help you make it more formal or more casual.


In [14]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [15]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [16]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [19]:
print(response.text)

[
  {
    "course": "data-engineering-zoomcamp",
    "course_name": "Data Engineering Zoomcamp",
    "path": "/json/data-engineering-zoomcamp.json",
    "questions_count": 404
  },
  {
    "course": "stock-markets-analytics-zoomcamp",
    "course_name": "Stock Markets Analytics Zoomcamp",
    "path": "/json/stock-markets-analytics-zoomcamp.json",
    "questions_count": 93
  },
  {
    "course": "ai-dev-tools-zoomcamp",
    "course_name": "AI Dev Tools Zoomcamp",
    "path": "/json/ai-dev-tools-zoomcamp.json",
    "questions_count": 41
  },
  {
    "course": "llm-zoomcamp",
    "course_name": "LLM Zoomcamp",
    "path": "/json/llm-zoomcamp.json",
    "questions_count": 85
  },
  {
    "course": "mlops-zoomcamp",
    "course_name": "MLOps Zoomcamp",
    "path": "/json/mlops-zoomcamp.json",
    "questions_count": 255
  },
  {
    "course": "machine-learning-zoomcamp",
    "course_name": "ML Zoomcamp",
    "path": "/json/machine-learning-zoomcamp.json",
    "questions_count": 472
  }
]


In [20]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [22]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [23]:
from minsearch import Index
 # text_fields are tokenized and ranked
 # tokenizing is like splitting text into words, lowercasing, removing stop words
 # keyword fields are for filtering
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

# fit an index on our documents like how in scikit-learn you fit a model on data
index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    # add more weight to the question field
    # add less weight to section field
    boost_dict={"question": 2.0, "section": 0.5},
    # filter by course
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [26]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)